# Logging Example

This notebook shows how to use the opt-in **logging** feature of `power-grid-model`.
Two logger types are available:

| Type | What it captures |
|------|------------------|
| `LoggerType.text` | Timestamped debug messages — useful for diagnosing sparse-matrix errors |
| `LoggerType.benchmark` | Per-phase timing totals — useful for performance analysis |

> **Important**: loggers provide *hints*, not conclusions. Output is intended for advanced
> users and may change between releases.
>
> Logging has a performance cost, which is why it is **opt-in**: no logger is active
> by default.

## Example Network

We reuse the 3-node network from the Power Flow Example:

```
 -----------------------line_8---------------
 |                                          |
node_1 ---line_3--- node_2 ----line_5---- node_6
 |                    |                     |
source_10          sym_load_4           sym_load_7
```

## Imports

In [1]:
import numpy as np

from power_grid_model import (
    AttributeType,
    CalculationMethod,
    ComponentType,
    DatasetType,
    LoadGenType,
    Logger,
    LoggerType,
    PowerGridModel,
    initialize_array,
)

## Build the Input Dataset

In [2]:
node = initialize_array(DatasetType.input, ComponentType.node, 3)
node[AttributeType.id] = [1, 2, 6]
node[AttributeType.u_rated] = [10.5e3, 10.5e3, 10.5e3]

line = initialize_array(DatasetType.input, ComponentType.line, 3)
line[AttributeType.id] = [3, 5, 8]
line[AttributeType.from_node] = [1, 2, 1]
line[AttributeType.to_node] = [2, 6, 6]
line[AttributeType.from_status] = [1, 1, 1]
line[AttributeType.to_status] = [1, 1, 1]
line[AttributeType.r1] = [0.25, 0.25, 0.25]
line[AttributeType.x1] = [0.2, 0.2, 0.2]
line[AttributeType.c1] = [10e-6, 10e-6, 10e-6]
line[AttributeType.tan1] = [0.0, 0.0, 0.0]
line[AttributeType.i_n] = [1000, 1000, 1000]

sym_load = initialize_array(DatasetType.input, ComponentType.sym_load, 2)
sym_load[AttributeType.id] = [4, 7]
sym_load[AttributeType.node] = [2, 6]
sym_load[AttributeType.status] = [1, 1]
sym_load[AttributeType.type] = [LoadGenType.const_power, LoadGenType.const_power]
sym_load[AttributeType.p_specified] = [20e6, 10e6]
sym_load[AttributeType.q_specified] = [5e6, 2e6]

source = initialize_array(DatasetType.input, ComponentType.source, 1)
source[AttributeType.id] = [10]
source[AttributeType.node] = [1]
source[AttributeType.status] = [1]
source[AttributeType.u_ref] = [1.0]

input_data = {
    ComponentType.node: node,
    ComponentType.line: line,
    ComponentType.sym_load: sym_load,
    ComponentType.source: source,
}

model = PowerGridModel(input_data)

## Text Logger

A `LoggerType.text` logger captures **human-readable diagnostic messages** emitted
during the calculation. Each line is timestamped and tagged with the event code.

Timer events (`log(tag)` with no payload) are recorded as bare event lines.
Numeric timing values are **not** captured — those belong exclusively to the
benchmark logger. String messages *are* captured; for example, the iterative
power-flow solver logs the **maximum voltage deviation** after every
Newton–Raphson iteration.

**Usage:** use `Logger` as a context manager. On entry the logger is registered on
the handle; on exit it is unregistered. The output buffer is preserved after the
`with` block and is accessible via `logger.output`.

In [ ]:
with Logger(LoggerType.text) as text_logger:
    output_data = model.calculate_power_flow(
        symmetric=True,
        calculation_method=CalculationMethod.newton_raphson,
    )

print(text_logger.output)

[2026-07-14 13:15:43.153Z] Tag:2100: 0.000043
[2026-07-14 13:15:43.153Z] Tag:2210: 0.000012
[2026-07-14 13:15:43.153Z] Tag:2221: 0.000016
[2026-07-14 13:15:43.153Z] Tag:2242: 0.000012
[2026-07-14 13:15:43.153Z] Tag:2225: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2226: 0.000012
[2026-07-14 13:15:43.153Z] Tag:2226: Iteration   1: max voltage deviation = 4.653741e-03 p.u.
[2026-07-14 13:15:43.153Z] Tag:2242: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2225: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2226: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2226: Iteration   2: max voltage deviation = 2.272320e-05 p.u.
[2026-07-14 13:15:43.153Z] Tag:2242: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2225: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2226: 0.000000
[2026-07-14 13:15:43.153Z] Tag:2226: Iteration   3: max voltage deviation = 5.460827e-10 p.u.
[2026-07-14 13:15:43.153Z] Tag:2227: 0.000024
[2026-07-14 13:15:43.153Z] Tag:2220: 0.000150
[2026-07-14 13:15:43.153Z] Tag:2246: 3
[2026-07-14 13:15:43.153Z] Tag:2200

## Benchmark Logger

A `LoggerType.benchmark` logger records **cumulative wall-clock time** spent in each
calculation phase. The output is tab-separated with two columns: `EVENT_CODE` and
`VALUE` (seconds).

Unlike the text logger, the benchmark logger ignores string messages entirely — it
only accumulates numeric (double) values from timer events.

In [ ]:
with Logger(LoggerType.benchmark) as bench_logger:
    output_data = model.calculate_power_flow(
        symmetric=True,
        calculation_method=CalculationMethod.newton_raphson,
    )

print(bench_logger.output)

2100	5.433e-06
2200	2.6964e-05
2220	2.3721e-05
2221	4.911e-06
2225	1.136e-06
2226	1.358e-06
2227	4.302e-06
2242	8.920000000000001e-07
2246	3
3000	1.0688e-05



### Parse the benchmark output into a DataFrame

The tab-separated format makes it easy to parse with `pandas`.

In [5]:
import io

import pandas as pd

bench_df = pd.read_csv(
    io.StringIO(bench_logger.output),
    sep="\t",
    header=None,
    names=["event_code", "time_s"],
)
bench_df = bench_df.sort_values("time_s", ascending=False)
print(bench_df.to_string(index=False))

 event_code       time_s
       2246 3.000000e+00
       2200 2.696400e-05
       2220 2.372100e-05
       3000 1.068800e-05
       2100 5.433000e-06
       2221 4.911000e-06
       2227 4.302000e-06
       2226 1.358000e-06
       2225 1.136000e-06
       2242 8.920000e-07


## Multiple Simultaneous Loggers

Multiple logger types can be combined in a single `with` statement. Each logger
independently captures its own output.

In [ ]:
with (
    Logger(LoggerType.text) as text_logger2,
    Logger(LoggerType.benchmark) as bench_logger2,
):
    output_data = model.calculate_power_flow(
        symmetric=True,
        calculation_method=CalculationMethod.newton_raphson,
    )

print("=== Text logger ===")
print(text_logger2.output)
print("=== Benchmark logger ===")
print(bench_logger2.output)

=== Text logger ===
[2026-07-14 13:15:46.781Z] Tag:2100: 0.000005
[2026-07-14 13:15:46.781Z] Tag:2221: 0.000004
[2026-07-14 13:15:46.781Z] Tag:2242: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2225: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2226: 0.000001
[2026-07-14 13:15:46.781Z] Tag:2226: Iteration   1: max voltage deviation = 4.653741e-03 p.u.
[2026-07-14 13:15:46.781Z] Tag:2242: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2225: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2226: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2226: Iteration   2: max voltage deviation = 2.272320e-05 p.u.
[2026-07-14 13:15:46.781Z] Tag:2242: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2225: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2226: 0.000000
[2026-07-14 13:15:46.781Z] Tag:2226: Iteration   3: max voltage deviation = 5.460827e-10 p.u.
[2026-07-14 13:15:46.781Z] Tag:2227: 0.000004
[2026-07-14 13:15:46.781Z] Tag:2220: 0.000025
[2026-07-14 13:15:46.781Z] Tag:2246: 3
[2026-07-14 13:15:46.781Z] Tag:2200: 0.000030
[2026-07-14 13:

## Accumulation and Clearing

Output accumulates across all calculations inside a single `with` block. Call
`logger.clear()` inside the block to reset between calculations without
unregistering.

In [ ]:
with Logger(LoggerType.benchmark) as acc_logger:
    # Run four calculations — output accumulates inside the with block
    for _ in range(4):
        model.calculate_power_flow(symmetric=True)

print(f"Combined output ({acc_logger.output.count(chr(10))} lines):")
print(acc_logger.output)

Combined output (10 lines):
2100	8.276e-06
2200	5.1832e-05
2220	4.2839e-05
2221	8.961e-06
2225	3.5739999999999996e-06
2226	2.7579999999999997e-06
2227	7.294e-06
2242	2.2159999999999996e-06
2246	3
3000	1.2516e-05



In [ ]:
# clear() resets the buffer without unregistering — useful between runs
with Logger(LoggerType.benchmark) as acc_logger:
    model.calculate_power_flow(symmetric=True)
    print(f"After first calc ({acc_logger.output.count(chr(10))} lines)")

    acc_logger.clear()
    print(f"After clear: {acc_logger.output!r}")

    model.calculate_power_flow(symmetric=True)

print(f"After second calc ({acc_logger.output.count(chr(10))} lines):")
print(acc_logger.output)

After clear, output is empty: ''
After fresh calculation (10 lines):
2100	1.2495e-05
2200	6.1358e-05
2220	5.3405e-05
2221	1.4257e-05
2225	1.075e-06
2226	3.4200000000000003e-06
2227	9.476e-06
2242	1.815e-06
2246	3
3000	6.136e-06



## Reuse Across Runs

A `Logger` object can be re-entered across multiple `with` blocks. Each entry
re-registers on the handle; each exit unregisters. Call `logger.clear()` between
runs to start with a fresh buffer.

In [ ]:
reuse_logger = Logger(LoggerType.benchmark)

with reuse_logger:
    model.calculate_power_flow(symmetric=True)
lines_run1 = reuse_logger.output.count("\n")

reuse_logger.clear()

with reuse_logger:
    model.calculate_power_flow(symmetric=True)
lines_run2 = reuse_logger.output.count("\n")

print(f"Run 1: {lines_run1} lines, Run 2: {lines_run2} lines")

[2026-07-14 13:15:48.894Z] Tag:2100: 0.000008
[2026-07-14 13:15:48.894Z] Tag:2221: 0.000006
[2026-07-14 13:15:48.894Z] Tag:2242: 0.000001
[2026-07-14 13:15:48.894Z] Tag:2225: 0.000001
[2026-07-14 13:15:48.894Z] Tag:2226: 0.000001
[2026-07-14 13:15:48.894Z] Tag:2226: Iteration   1: max voltage deviation = 4.653741e-03 p.u.
[2026-07-14 13:15:48.894Z] Tag:2242: 0.000000
[2026-07-14 13:15:48.894Z] Tag:2225: 0.000000
[2026-07-14 13:15:48.894Z] Tag:2226: 0.000000
[2026-07-14 13:15:48.894Z] Tag:2226: Iteration   2: max voltage deviation = 2.272320e-05 p.u.
[2026-07-14 13:15:48.894Z] Tag:2242: 0.000000
[2026-07-14 13:15:48.894Z] Tag:2225: 0.000000
[2026-07-14 13:15:48.894Z] Tag:2226: 0.000000
[2026-07-14 13:15:48.894Z] Tag:2226: Iteration   3: max voltage deviation = 5.460827e-10 p.u.
[2026-07-14 13:15:48.894Z] Tag:2227: 0.000030
[2026-07-14 13:15:48.894Z] Tag:2220: 0.000064
[2026-07-14 13:15:48.894Z] Tag:2246: 3
[2026-07-14 13:15:48.894Z] Tag:2200: 0.000075
[2026-07-14 13:15:48.894Z] Tag:3000

## Python Logging Integration

Pass `python_logger=` to route output through the standard `logging` module.
On `__exit__`, accumulated lines are flushed to `python_logger` — one log record
per line at the configured level (default `logging.DEBUG`) — and the buffer is
cleared. No manual flush needed.

In [ ]:
import logging

# Configure the standard logging module to print to stdout for this demo.
logging.basicConfig(level=logging.DEBUG, format="%(levelname)s %(name)s: %(message)s")
py_logger = logging.getLogger("power_grid_model")

with Logger(LoggerType.text, python_logger=py_logger) as py_log_logger:
    model.calculate_power_flow(
        symmetric=True,
        calculation_method=CalculationMethod.newton_raphson,
    )

# __exit__ flushed each line to py_logger automatically; buffer is now empty
print(f"Buffer after exit: {py_log_logger.output!r}")

DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2100: 0.000006
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2221: 0.000006
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2242: 0.000001
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2225: 0.000000
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2226: 0.000002
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2226: Iteration   1: max voltage deviation = 4.653741e-03 p.u.
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2242: 0.000000
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2225: 0.000000
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2226: 0.000000
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2226: Iteration   2: max voltage deviation = 2.272320e-05 p.u.
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2242: 0.000000
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2225: 0.000000
DEBUG power_grid_model: [2026-07-14 13:15:50.085Z] Tag:2226: 0.0

Buffer after flush: ''


## Batch Calculations

Loggers work transparently with batch calculations. Each batch scenario's
internal thread contributes to the same logger via the composite logger
mechanism, so the output contains events from all scenarios.

In [ ]:
# Create a small batch update: vary the load in node_2 across 3 scenarios
load_update = initialize_array(DatasetType.update, ComponentType.sym_load, (3, 3))
load_update[AttributeType.id] = [4, 4, 4]
load_update[AttributeType.p_specified] = [10e6, 20e6, 30e6]
update_data = {ComponentType.sym_load: load_update}

with Logger(LoggerType.benchmark) as batch_bench:
    batch_output = model.calculate_power_flow(
        update_data=update_data,
        symmetric=True,
        threading=0,  # parallel
    )

print(batch_bench.output)

64	0.000134955
128	0.00019802600000000002
1100	5.9969e-05
1200	3.4789e-05
1201	3.559e-06
2100	2.0821e-05
2200	8.5789e-05
2220	7.0485e-05
2221	2.4050000000000002e-05
2225	4.177e-06
2226	3.927e-06
2227	1.0501e-05
2242	3.2369999999999997e-06
2246	3
3000	1.4789e-05



## Safety Notes

- **Do not** destroy (or let Python GC collect) a `Logger` inside an active `with` block — this
  leaves a dangling pointer in the C layer and triggers a `ResourceWarning`.
- **Do not** use the same logger from multiple user threads simultaneously (internal C-level
  batch threads are safe).
- Registering the same logger to the same handle more than once is a **no-op** (idempotent).
- Accessing `logger.output` or calling `logger.clear()` on a `LoggerType.do_nothing` logger is
  always safe — it returns `""` and is a no-op respectively.